# 06. 한국어 텍스트 → 음성(TTS)

**실습 목표**  
한국어 문장을 Hugging Face의 사전학습 TTS 모델로 음성 파형으로 변환한다.

**주요 Hugging Face 모델**  
`facebook/mms-tts-kor`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [20]:
# uv add transformers accelerate scipy

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

PyTorch: 2.14.0+cu126
CUDA available: True
device: cuda


In [2]:
from transformers import pipeline
from IPython.display import Audio
import torch

DEVICE = 0 if torch.cuda.is_available() else -1

tts = pipeline(
    "text-to-speech",
    model="nineninesix/kani-tts-400m-ko",
    device=DEVICE
)

text = "안녕하세요. 허깅페이스 멀티모달 인공지능 실습을 시작합니다."

speech = tts(text)

print(speech.keys())

config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  740MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/7.88M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


dict_keys(['audio', 'sampling_rate'])


In [3]:
print(speech.keys())

dict_keys(['audio', 'sampling_rate'])


In [4]:
print(speech.keys())
print("sampling_rate:", speech.get("sampling_rate"))
print("audio type:", type(speech["audio"]))
print("audio shape:", speech["audio"].shape)


dict_keys(['audio', 'sampling_rate'])
sampling_rate: None
audio type: <class 'numpy.ndarray'>
audio shape: (283,)


In [5]:
print(tts.model.config)

Lfm2Config {
  "_from_model_config": true,
  "architectures": [
    "Lfm2ForCausalLM"
  ],
  "block_auto_adjust_ff_dim": true,
  "block_dim": 1024,
  "block_ffn_dim_multiplier": 1.0,
  "block_mlp_init_scale": 1.0,
  "block_multiple_of": 256,
  "block_norm_eps": 1e-05,
  "block_out_init_scale": 1.0,
  "block_use_swiglu": true,
  "block_use_xavier_init": true,
  "bos_token_id": 1,
  "conv_L_cache": 3,
  "conv_bias": false,
  "conv_dim": 1024,
  "conv_dim_out": 1024,
  "conv_use_xavier_init": true,
  "dtype": "bfloat16",
  "eos_token_id": 7,
  "full_attn_idxs": null,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 6656,
  "layer_types": [
    "conv",
    "conv",
    "full_attention",
    "conv",
    "conv",
    "full_attention",
    "conv",
    "conv",
    "full_attention",
    "conv",
    "full_attention",
    "conv",
    "full_attention",
    "conv",
    "full_attention",
    "conv"
  ],
  "max_position_embeddings": 128000,
  "model_type": "lfm2",
  "norm_eps"

In [7]:
Audio(
    speech["audio"],
    rate=22000
)

In [8]:
import scipy.io.wavfile as wavfile
import numpy as np
import os

os.makedirs("output", exist_ok=True)

audio = np.asarray(speech["audio"]).squeeze()

wavfile.write(
    "output/korean_tts.wav",
    22000,
    audio
)

# 

## 안정적 실행

In [ ]:
# uv remove transformers
# uv remove protobuf

# uv add kani-tts   <-- 직접실행해서 해야함.
# uv add "transformers==4.57.1"